#  1. Business Question

<span style="color:red"> ##### Where are delays happening, what appears to cause them, and what should RouteReady fix first?</span>

# 2. Data Understanding

#### Import Libraries

In [1]:
#Import the Libraries
import pandas as pd

#### Load the Data

In [5]:
shipments = pd.read_csv("data/shipments.csv")
customers = pd.read_csv("data/customers.csv")
depots = pd.read_csv("data/depots.csv")
daily_ops = pd.read_csv("data/daily_depot_operations.csv")

#### Display the first few rows

In [64]:
#Display the first few rows
datasets = {
    "Shipments": shipments,
    "Customers": customers,
    "Depots": depots,
    "Daily Depot Operations": daily_ops
}

for name, df in datasets.items():
    print("=" * 50)
    print(f"{name}")
    print("=" * 50)
    display(df.head())

Shipments


,shipment_id,ship_date,customer_id,depot_id,service_level,package_type,distance_km,weight_kg,weekend_pickup,weather_condition,promised_delivery_days,actual_delivery_days,delayed,shipping_cost_usd,customer_rating,delay_reason
0,S02742,2025-06-17,C0252,D02,Standard,Box,743.0,10.8,0,Clear,3,4,1,49.43,NaN,Route traffic
1,S02340,2025-05-14,C0240,D06,Economy,Box,412.4,29.4,0,Clear,5,5,0,40.36,5.0,NaN
2,S02125,2025-02-24,C0089,D03,Express,Box,571.9,10.3,0,Snow,1,2,1,44.67,NaN,Depot capacity
3,S01837,2025-06-18,C0273,D05,Standard,Box,157.7,7.5,0,Clear,3,3,0,28.69,3.0,NaN
4,S02355,2025-03-24,C0020,D06,Standard,Satchel,868.2,1.4,0,Rain,3,5,1,34.66,2.0,Depot capacity


Customers


,customer_id,customer_segment,customer_region,account_age_months,average_monthly_shipments
0,C0001,Small Business,Central,68,24
1,C0002,Enterprise,North,31,146
2,C0003,Retail,Central,76,53
3,C0004,Enterprise,West,65,166
4,C0005,Retail,Central,3,57


Depots


,depot_id,depot_name,depot_region,automation_level,processing_capacity_per_day,typical_staff_on_shift
0,D01,Metro North Hub,North,High,120,30
1,D02,South Gateway,South,Medium,105,26
2,D03,Central Sort Center,Central,Low,105,24
3,D04,East Regional Hub,East,High,125,31
4,D05,Mountain Depot,West,Medium,90,21


Daily Depot Operations


,date,depot_id,packages_received,packages_processed,backlog_packages,overtime_hours,staff_absent,avg_sort_time_minutes
0,2025-01-01,D01,111,111,0,0.0,0,34.5
1,2025-01-01,D02,87,87,0,1.3,2,37.2
2,2025-01-01,D03,122,111,11,8.6,3,54.0
3,2025-01-01,D04,100,100,0,0.0,2,30.7
4,2025-01-01,D05,70,70,0,2.6,3,35.8


#### Report the Number of Rows and Columns

In [65]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")


Shipments
Rows: 3012
Columns: 16

Customers
Rows: 300
Columns: 5

Depots
Rows: 6
Columns: 6

Daily Depot Operations
Rows: 1086
Columns: 8


#### Review Column Names and Data Types

In [66]:
#Review Column Names and data Types
for name, df in datasets.items():
    print("=" * 20)
    print(f"{name}")
    print("=" * 20)
    column_info = pd.DataFrame({
        "Column Name": df.columns,
        "Data Type": df.dtypes.values
    })

    display(column_info)

Shipments


,Column Name,Data Type
0,shipment_id,object
1,ship_date,object
2,customer_id,object
3,depot_id,object
4,service_level,object
5,package_type,object
6,distance_km,float64
7,weight_kg,float64
8,weekend_pickup,int64
9,weather_condition,object


Customers


,Column Name,Data Type
0,customer_id,object
1,customer_segment,object
2,customer_region,object
3,account_age_months,int64
4,average_monthly_shipments,int64


Depots


,Column Name,Data Type
0,depot_id,object
1,depot_name,object
2,depot_region,object
3,automation_level,object
4,processing_capacity_per_day,int64
5,typical_staff_on_shift,int64


Daily Depot Operations


,Column Name,Data Type
0,date,object
1,depot_id,object
2,packages_received,int64
3,packages_processed,int64
4,backlog_packages,int64
5,overtime_hours,float64
6,staff_absent,int64
7,avg_sort_time_minutes,float64


#### Count Missing Values

In [67]:
#Count Missing Values
for name, df in datasets.items():
    print("=" * 20)
    print(f"{name}")
    print("=" * 20)

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print(missing)
    else:
        print("No missing values")

Shipments
weight_kg              45
weather_condition      30
customer_rating       513
delay_reason         1942
dtype: int64
Customers
No missing values
Depots
No missing values
Daily Depot Operations
No missing values


#### Check Duplicate Rows and Duplicate Keys

In [68]:
#Check Duplicate Rows and Duplicate Keys

datasets = {
    "Shipments": (shipments, "shipment_id"),
    "Customers": (customers, "customer_id"),
    "Depots": (depots, "depot_id"), 
    "Daily Depot Operation": (daily_ops, ["depot_id", "date"])
}

for name, (df, key) in datasets.items():
    print(f"\n{'='*50}")
    print(name)
    print(f"{'='*50}")

    # Duplicate rows
    print(f"Duplicate Rows: {df.duplicated().sum()}")

    # Duplicate keys
    print(f"Duplicate {key} Values: {df[key].duplicated().sum()}")


    


Shipments
Duplicate Rows: 12
Duplicate shipment_id Values: 12

Customers
Duplicate Rows: 0
Duplicate customer_id Values: 0

Depots
Duplicate Rows: 0
Duplicate depot_id Values: 0

Daily Depot Operation
Duplicate Rows: 0
Duplicate ['depot_id', 'date'] Values: 0


####  Table Grain

- **shipments.csv**
Each row represents a single shipment, including its characteristics, delivery outcome, cost, customer rating, and delay information.

- **customers.csv**
Each row represents a single customer and their associated attributes, such as segment, region, account age, and shipment volume.

- **depots.csv**
Each row represents a single depot and its operational characteristics, including staffing, capacity, automation level, and region.

- **daily_depot_operations.csv**
Each row represents the daily operational metrics for one depot on a specific date.

#### What is the primary key of each table?
- **shipments.csv:**  shipment_id
- **customers.csv:** customer_id
- **depots.csv:** depot_id
- **daily_depot_operation.csv:** composite key (depot_id, date)

#### Which columns connect the tables?
```mermaid
erDiagram
    customer ||--o{ shipments : customer_id
    depots ||--o{ shipments : depot_id

    daily_depot_operations ||--o{ shipments : depot_id_and_date  
```

#### Information Known Before Delivery


In [ ]:
# The "Customers" dataset is entirely information that would be known before the delivery. "Depots" is mixed, as well as is "Shipments". Within the "Shipments" data, the follwowing is before: 
#     shipment_id	ship_date	customer_id	depot_id	service_level	package_type	distance_km	weight_kg	weekend_pickup    shipping_cost_usd
#     weather_condition could be a before or after data point.
# Within "Depots", the following is before: 
#     depot_id	depot_name	depot_region	automation_level

#### Outcomes Recorded After Delivery

In [ ]:
# The "Daily Depot Operations" dataset is entirely after. The "Depots" is mixed, as well as is "Shipments". Within "Shipments" data, the following is after: The "Depots" dataand "Daily Depot Operations" 
#     promised_delivery_days	actual_delivery_days	delayed,	customer_rating	delay_reason
# Within "Depots", the following is after: 
#     processing_capacity_per_day	typical_staff_on_shift

#### Printed Summary  <span style="color:red"> *** TO BE REVISITED

In [7]:
# Ran tests on data shape. Only concerns arose in shipments...
# Combine shipment results from "Count Missing Values" and "Check Duplicate Rows" to here. 

#### Relationship Explanation  <span style="color:red">

In [ ]:
# The "customers", "depots", and "daily_depot_operations" flow into the "shipments" data set. They are all related via the unique id's that each dataset has. *** TO BE REVISITED

# 3. Data Cleaning

#### Remove Exact Duplicate Rows

In [69]:
#Check Rows Before Cleaning
print(f"Rows before cleaning: {shipments.shape[0]}")

#Remove Duplicate Rows
shipments_clean=shipments.copy()
shipments_clean=shipments_clean.drop_duplicates()

print(f"Rows after cleaning: {shipments_clean.shape[0]}")
print(f"Duplicates removed: {shipments.shape[0] - shipments_clean.shape[0]}")

Rows before cleaning: 3012
Rows after cleaning: 3000
Duplicates removed: 12


#### Convert `ship_date` to a datetime data type

In [70]:
#Convert Ship Date to Datetime
shipments_clean["ship_date"] = pd.to_datetime(
    shipments_clean["ship_date"]
)
shipments_clean["ship_date"].dtype

dtype('<M8[ns]')

#### Standarized Service Level

In [71]:
print(sorted(shipments["service_level"].unique()))

shipments_clean["service_level"]=(
    shipments_clean["service_level"]
    .str.strip()
    .str.title()
)
print(sorted(shipments_clean["service_level"].unique()))


[' Economy ', ' Express ', ' Standard ', 'ECONOMY', 'EXPRESS', 'Economy', 'Express', 'STANDARD', 'Standard', 'economy', 'express', 'standard']
['Economy', 'Express', 'Standard']


#### Standarized Package Type

In [72]:
print(sorted(shipments["package_type"].unique()))

shipments_clean["package_type"]=(
   shipments_clean["package_type"]
  .str.strip()
 .str.title()
)
print(sorted(shipments_clean["package_type"].unique()))

[' Box ', ' Oversize ', ' Satchel ', 'BOX', 'Box', 'OVERSIZE', 'Oversize', 'SATCHEL', 'Satchel', 'box', 'oversize', 'satchel']
['Box', 'Oversize', 'Satchel']


#### Handle missing `weight_kg` values

In [73]:
print(f"Missing value before cleaning:  {shipments['weight_kg'].isna().sum()}")

shipments_clean["weight_kg"]=(
    shipments_clean.groupby("package_type")["weight_kg"].transform(lambda x: x.fillna(x.median()))
)

print(f"Missing value after cleaning: {shipments_clean['weight_kg'].isna().sum()}")

Missing value before cleaning:  45
Missing value after cleaning: 0


#### Handle Missing `weather_condition` Values

In [74]:
print(f"Missing counts before cleaning: {shipments['weather_condition'].isna().sum()}")

shipments_clean["weather_condition"] = (
    shipments_clean["weather_condition"]
    .fillna("Unknown")
)

print(f"Missing counts after cleaning: {shipments_clean['weather_condition'].isna().sum()}")

Missing counts before cleaning: 30
Missing counts after cleaning: 0


####  Keep Missing Customer Ratings  <span style="color:red"> *** Need Review </span>
Missing customer rating were left unchanged. 

A missing rating indicates that the customer did not complete the survey and should not be interpreted as dissatisfaction. 

#### Verify the `delayed` Flag

In [75]:
calculated_delay = (
    shipments_clean["actual_delivery_days"]
    > shipments_clean["promised_delivery_days"]
).astype(int)

mismatch_count = (
    shipments_clean["delayed"]
    != calculated_delay
).sum()

print("Mismatched delayed flags:", mismatch_count)

Mismatched delayed flags: 0


# 4. Data Integration

#### Join Cleaned Shipments to `customers.csv`

In [76]:
shipments_customers = shipments_clean.merge(
    customers,
    on="customer_id",
    how="left"
)

##### Check the Row Counts

In [77]:
print("Rows before join:", shipments_clean.shape[0])
print("Rows after join:", shipments_customers.shape[0])

Rows before join: 3000
Rows after join: 3000


##### Check Unmatched Keys

In [78]:
unmatched_customers = shipments_customers[
    shipments_customers["customer_id"].isna()
]

print("Unmatched customers:", len(unmatched_customers))

Unmatched customers: 0


##### Confirm One Shipment Appears Once

In [79]:
duplicate_shipments = (
    shipments_customers["shipment_id"]
    .duplicated()
    .sum()
)

print("Duplicate shipment IDs:", duplicate_shipments)

Duplicate shipment IDs: 0


#### Join to Depots

In [80]:
analysis_table = shipments_customers.merge(
    depots,
    on="depot_id",
    how="left"
)

##### Check the Row Counts

In [81]:
print("Rows before join:", shipments_customers.shape[0])
print("Rows after join:", analysis_table.shape[0])

Rows before join: 3000
Rows after join: 3000


##### Check Unmatched Keys

In [82]:
unmatched_depots = analysis_table[
    shipments_customers["customer_id"].isna()
]

print("Unmatched customers:", len(unmatched_customers))

Unmatched customers: 0


##### Confirm One Shipment Appears Once

In [83]:
duplicate_shipments = (
    analysis_table["shipment_id"]
    .duplicated()
    .sum()
)

print("Duplicate shipment IDs:", duplicate_shipments)

Duplicate shipment IDs: 0


# 5. Exploratory Data Analysis (EDA) and Statistics

#### Overall Delay Rate

In [84]:
delay_rate = analysis_table["delayed"].mean()

print(f"Overall Delay Rate: {delay_rate:.2%}")

Overall Delay Rate: 35.50%


#### Delay Rate by Depot    *** Add Percentage

In [85]:
depot_delay = (
    analysis_table
    .groupby("depot_id")["delayed"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

depot_delay

,depot_id,delayed
0,D03,0.607383
1,D05,0.396135
2,D06,0.308617
3,D02,0.291667
4,D01,0.248047
5,D04,0.239006


#### Delay Rate by Service Level

In [92]:
service_delay = (
    analysis_table
    .groupby("service_level")["delayed"]
    .mean()
    .sort_values(ascending=False)
     .reset_index()
)

service_delay

,service_level,delayed
0,Economy,0.393443
1,Standard,0.367105
2,Express,0.273163


#### Delay Rate by Weather Condition

In [93]:
weather_delay = (
    analysis_table
    .groupby("weather_condition")["delayed"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

weather_delay

,weather_condition,delayed
0,Storm,0.569307
1,Snow,0.547059
2,Rain,0.357823
3,Clear,0.314546
4,Unknown,0.266667


#### Delay Rate by Package Type

In [94]:
package_delay = (
    analysis_table
    .groupby("package_type")["delayed"]
    .mean()
    .sort_values(ascending=False)
     .reset_index()
)

package_delay

,package_type,delayed
0,Oversize,0.547804
1,Satchel,0.343447
2,Box,0.318614


#### Delay Rate by Customer Segment

In [96]:
segment_delay = (
    analysis_table
    .groupby("customer_segment")["delayed"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

segment_delay

,customer_segment,delayed
0,Retail,0.381089
1,Small Business,0.343826
2,Enterprise,0.340260


#### Compare Delayed vs On-Time Shipments

##### Distance

In [97]:
analysis_table.groupby("delayed")["distance_km"].agg(
    ["mean", "median"]
)

,mean,median
delayed,,
0,684.373850,603.7
1,792.780751,688.3


##### Weight

In [98]:
analysis_table.groupby("delayed")["weight_kg"].agg(
    ["mean", "median"]
)

,mean,median
delayed,,
0,9.659690,6.9
1,12.386667,7.7


##### Shipment Cost

In [100]:
analysis_table.groupby("delayed")["shipping_cost_usd"].agg(
    ["mean", "median"]
)

,mean,median
delayed,,
0,44.831514,41.39
1,50.367202,45.98


##### Combined View (Distance, Weight, and Shipment cost differ between delayed and on-time shipments)

In [102]:
analysis_table.groupby("delayed")[
    ["distance_km",
     "weight_kg",
     "shipping_cost_usd"]
].mean()

,distance_km,weight_kg,shipping_cost_usd
delayed,,,
0,684.373850,9.659690,44.831514
1,792.780751,12.386667,50.367202


#### Customer Rating : Delayed vs On-Time Shipments

In [104]:
analysis_table.groupby("delayed")[
    "customer_rating"
].agg(["count","mean","median"])

,count,mean,median
delayed,,,
0,1606,4.385430,4.0
1,882,2.621315,3.0


#### Monthly Delay Rate (January to June)

In [107]:
analysis_table["month"] = (
    analysis_table["ship_date"]
    .dt.month_name()
)
month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June"
]

monthly_delay = (
    analysis_table
    .groupby("month")["delayed"]
    .mean()
    .reindex(month_order)
)

monthly_delay


month
January     0.331313
February    0.371638
March       0.332661
April       0.344622
May         0.344887
June        0.406910
Name: delayed, dtype: float64

#### Most Common Delay Reason

In [110]:
delay_reason_counts = (
    analysis_table["delay_reason"]
    .dropna()
    .value_counts()
)

delay_reason_counts


delay_reason
Route traffic       384
Depot capacity      305
Weather             196
Weekend handoff     103
Special handling     77
Name: count, dtype: int64

#### Mean and Median Shipment Cost

In [112]:
cost_mean = analysis_table["shipping_cost_usd"].mean()
cost_median = analysis_table["shipping_cost_usd"].median()

print("Mean Cost:", round(cost_mean,2))
print("Median Cost:", round(cost_median,2))

Mean Cost: 46.8
Median Cost: 42.82


#### Mean and Median Actual Delivery Days

In [113]:
delivery_mean = analysis_table["actual_delivery_days"].mean()
delivery_median = analysis_table["actual_delivery_days"].median()

print("Mean Delivery Days:", round(delivery_mean,2))
print("Median Delivery Days :", round(delivery_median,2))

Mean Delivery Days: 3.6
Median Delivery Days : 3.0


#### Standard Deviation of Shipment Cost

In [114]:
cost_std = analysis_table["shipping_cost_usd"].std()

print("STD of Shipment Cost:", round(cost_std,2))

STD of Shipment Cost: 19.8


#### Correlation Analysis

##### Weight vs Cost

In [115]:
weight_cost_corr = (
    analysis_table["weight_kg"]
    .corr(analysis_table["shipping_cost_usd"])
)

print("Weight vs Cost Correlation:",
      round(weight_cost_corr,3))

Weight vs Cost Correlation: 0.707


##### Distance vs Cost

In [116]:
distance_cost_corr = (
    analysis_table["distance_km"]
    .corr(analysis_table["shipping_cost_usd"])
)

print("distance vs Cost Correlation:",
      round(distance_cost_corr,3))

distance vs Cost Correlation: 0.609
